# Week 5 — Multi-Dimensional Parallelism (3D) and Large-Cluster Scaling

**Course:** High-Performance Computing & Scaling Large Models  
**Practical:** Tensor and Pipeline parallelism with Megatron-LM.

---

## Learning Objectives

1. Distinguish the four canonical parallelism axes: **data**, **tensor**, **pipeline**, and **sequence/context**.
2. Derive the communication volume of each and reason about which interconnects they should occupy.
3. Implement a *toy* tensor-parallel `Linear` layer that reproduces Megatron-LM's column/row split.
4. Understand pipeline scheduling (GPipe, 1F1B, interleaved) and the *bubble* fraction.
5. Combine the three axes — **3D parallelism** — into the topology used to train GPT-3 and PaLM.
6. Read a Megatron-LM launch script and identify each parallelism degree.


## 1. Why One Axis Is Not Enough

ZeRO and FSDP scale memory in the data-parallel dimension. They cannot, however, exceed the *per-GPU compute* available at a fixed batch size: a forward pass through a single transformer block on a single GPU at sequence length 4096 takes a fixed amount of time. Beyond a certain point, adding GPUs to the data-parallel group reduces per-step latency only by the (small) amount of communication overlap.

For models above ~30B parameters, the *weight* of a single layer's matmul (e.g., a $4096 \times 16384$ FFN matrix) is itself the bottleneck. We need to split the *computation* — not just the storage — across devices.

Three orthogonal axes accomplish this:

| Axis | What is sharded | Communication primitive | Frequency |
|------|----------------|--------------------------|-----------|
| **Data Parallelism** (DP) | Batch | `all-reduce` of gradients | 1 × per step |
| **Tensor Parallelism** (TP) | Hidden dimension within a layer | `all-reduce` of activations | 2 × per layer |
| **Pipeline Parallelism** (PP) | Layers across devices | Point-to-point activations | 1 × per stage boundary |
| **Sequence / Context** (SP / CP) | Sequence dimension | `all-gather` of K/V | 1 × per attention |

Each axis has a distinct *communication signature* — and the punchline of 3D parallelism is that we route each kind of traffic onto the interconnect best suited for it.


## 2. Tensor Parallelism: Splitting the Matmul

Consider a linear layer $Y = XA$ where $A \in \mathbb{R}^{d \times h}$. Megatron-LM (Shoeybi et al., 2019) observes that a transformer's two adjacent linears in the MLP can be split *complementarily*:

- **Column-parallel** ($A = [A_1, A_2]$): each GPU computes $Y_i = X A_i$, holds an output *slice*.
- **Row-parallel** ($B = [B_1; B_2]$, $X = [X_1, X_2]$): each GPU computes $Z_i = X_i B_i$, then `all-reduce` to sum.

If we pair a column-parallel layer with a row-parallel layer, the GELU activation between them is computed independently on each GPU on its slice — no communication. The only `all-reduce` happens at the end of the MLP block.

For multi-head attention, the same trick applies along the head dimension: heads are partitioned across TP ranks, the attention is computed per-shard, and the output projection is row-parallel.

### Communication Volume

For a TP group of size $T$, one transformer block exchanges $2 \cdot B \cdot S \cdot d$ bytes (per direction). This must happen *every layer*, so TP traffic dominates the interconnect — which is why TP is almost always confined to *within a node*, on NVLink (600 GB/s on A100, 900 GB/s on H100). Pipeline and data parallelism, with lower-frequency communication, take the slower inter-node links.

### Implementation Sketch

The cell below implements a column-parallel linear layer using PyTorch's distributed primitives. It runs in a 1-GPU "world" (effectively a no-op) and is fully correct when launched under `torchrun`.


In [ ]:
import os, math, time
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.distributed as dist

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
print(f"Distributed initialized: {dist.is_available() and dist.is_initialized()}")


In [ ]:
# Toy column-parallel linear. In production, use megatron.core.tensor_parallel.
class ColumnParallelLinear(nn.Module):
    """
    Y = X @ A,  with A = [A_1 | A_2 | ... | A_T] split along the *output* dim.
    Each TP rank holds A_i locally and produces the i-th slice of Y.

    `gather_output=True` collects all slices into a full Y (extra all-gather).
    `gather_output=False` returns the local slice — the typical Megatron usage.
    """
    def __init__(self, in_features, out_features, tp_size=1, tp_rank=0,
                 gather_output=False, bias=True):
        super().__init__()
        assert out_features % tp_size == 0, "out_features must be divisible by tp_size"
        self.tp_size = tp_size
        self.tp_rank = tp_rank
        self.gather_output = gather_output

        self.local_out = out_features // tp_size
        # Each rank owns out_features // tp_size columns of A.
        self.weight = nn.Parameter(torch.empty(self.local_out, in_features))
        nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))
        self.bias = nn.Parameter(torch.zeros(self.local_out)) if bias else None

    def forward(self, x):
        y = F.linear(x, self.weight, self.bias)
        if self.gather_output and self.tp_size > 1:
            # Gather slices across the TP group → reconstruct full y
            chunks = [torch.empty_like(y) for _ in range(self.tp_size)]
            dist.all_gather(chunks, y)
            y = torch.cat(chunks, dim=-1)
        return y


class RowParallelLinear(nn.Module):
    """
    Y = X @ B,  with B = [B_1; B_2; ...; B_T] split along the *input* dim.
    Each rank holds the i-th block of rows; partial sums are all-reduced.

    Input X is assumed to be already partitioned along the last dim
    (it is the output of a ColumnParallel layer).
    """
    def __init__(self, in_features, out_features, tp_size=1, tp_rank=0, bias=True):
        super().__init__()
        assert in_features % tp_size == 0, "in_features must be divisible by tp_size"
        self.tp_size = tp_size
        self.tp_rank = tp_rank

        self.local_in = in_features // tp_size
        self.weight = nn.Parameter(torch.empty(out_features, self.local_in))
        nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))
        self.bias = nn.Parameter(torch.zeros(out_features)) if bias else None

    def forward(self, x):
        # x is the local slice of an earlier ColumnParallel output
        y = F.linear(x, self.weight, None)
        if self.tp_size > 1:
            dist.all_reduce(y, op=dist.ReduceOp.SUM)
        if self.bias is not None:
            y = y + self.bias
        return y


# Smoke test: tp_size=1 reduces to vanilla nn.Linear behaviour
torch.manual_seed(0)
col = ColumnParallelLinear(64, 128, tp_size=1, gather_output=False, bias=False)
row = RowParallelLinear(128, 64, tp_size=1, bias=False)
x   = torch.randn(2, 16, 64)
y   = row(col(x))
print(f"Output shape: {tuple(y.shape)}  (expected (2, 16, 64))")


**Reference Megatron-LM layout for a transformer MLP block:**

```
                                 │ TP rank 0     TP rank 1
─────────────────────────────────┼───────────────────────────
Linear (d → 4d)   column-parallel│  W₀: (d × 2d)  W₁: (d × 2d)
GELU                             │  per-rank      per-rank
Linear (4d → d)   row-parallel   │  V₀: (2d × d)  V₁: (2d × d)
  → all-reduce at the end        │ ───────► sum ◄───────
```

A *single* all-reduce per MLP block (and another per attention block). All the GELU/LayerNorm work happens in parallel on slices.


## 3. Pipeline Parallelism: Splitting Across Layers

A 100-layer model whose individual layers fit on one GPU can be split *vertically*: layers 0–24 on GPU 0, 25–49 on GPU 1, etc. The forward pass becomes a chain — micro-batch `m` flows from GPU 0 to GPU 3, then its loss flows backward in reverse.

### The Bubble

A naive (GPipe) schedule processes one micro-batch end-to-end before starting the next. With $P$ pipeline stages and $M$ micro-batches per global batch, the *bubble fraction* (idle time) is

$$
\text{bubble} = \frac{P - 1}{M + P - 1}.
$$

For $P=4, M=4$ the bubble is 43 %. For $P=4, M=32$ it drops to 9 %. **Pipeline efficiency improves with smaller micro-batches** — which is the opposite of what data parallelism wants.

### The 1F1B Schedule

Narayanan et al. (2021) introduced *1-Forward-1-Backward (1F1B)* scheduling: as soon as a GPU finishes the forward pass of micro-batch $m$, it begins the backward pass of an earlier micro-batch $m'$. Activations are held only for the in-flight micro-batches, capping peak memory at $O(P)$ instead of $O(M)$.

**Interleaved 1F1B** further chops each pipeline stage into *virtual* sub-stages, reducing the bubble at the cost of extra inter-stage communication. This is Megatron's default for the largest models.

### Visualizing the Schedule

The chart below contrasts naive and 1F1B for $P=4, M=8$. Cells are colored by which micro-batch is running where.


In [ ]:
# Schedule visualization (offline; no distributed runtime needed)
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import matplotlib.colors as mcolors

def gpipe_schedule(P, M):
    """Naive GPipe: do all forwards, then all backwards. Returns 2D array."""
    n_slots = 2 * (M + P - 1)
    sched = np.full((P, n_slots), -1, dtype=int)  # -1 means idle
    # Forwards: micro-batch m starts on stage 0 at slot m, reaches stage p at m+p
    for m in range(M):
        for p in range(P):
            sched[p, m + p] = m
    # Backwards: begin after all forwards complete
    bwd_start = M + P - 1
    for m in range(M):
        for p in range(P):
            # Backward goes in reverse stage order
            sched[P - 1 - p, bwd_start + m + p] = -100 - m   # encode as backward
    return sched


def onefonefb_schedule(P, M):
    """Simplified 1F1B. Forward warm-up, then alternating F/B, then cooldown."""
    n_slots = 2 * (M + P - 1)
    sched = np.full((P, n_slots), -1, dtype=int)
    # Phase 1: forward warm-up — fill the pipe
    for m in range(M):
        for p in range(P):
            sched[p, m + p] = m
    # Phase 2: 1F1B steady state — backward of (m - P + 1) interleaved with forward of m
    # Phase 3: backward drain
    bwd_idx = 0
    for m in range(M):
        for p in range(P):
            # backward of micro-batch m on stage (P-1-p) at time M + p + m
            t = (M - 1) + m + p
            if t < n_slots:
                sched[P - 1 - p, t] = -100 - m
    return sched


def plot(sched, ax, title):
    P, T = sched.shape
    cmap = plt.get_cmap("tab20")
    for p in range(P):
        for t in range(T):
            v = sched[p, t]
            if v == -1:
                color = "#eeeeee"
                txt   = ""
            elif v >= 0:
                color = cmap(v % 20)
                txt   = f"F{v}"
            else:
                color = cmap((-100 - v) % 20)
                color = mcolors.to_rgba(color)
                color = (color[0]*0.6, color[1]*0.6, color[2]*0.6, 1.0)
                txt   = f"B{-100 - v}"
            ax.add_patch(plt.Rectangle((t, P - 1 - p), 1, 1,
                         facecolor=color, edgecolor="white"))
            if txt:
                ax.text(t + 0.5, P - 1 - p + 0.5, txt, ha="center",
                        va="center", fontsize=7)
    ax.set_xlim(0, T)
    ax.set_ylim(0, P)
    ax.set_yticks([i + 0.5 for i in range(P)])
    ax.set_yticklabels([f"GPU {P - 1 - i}" for i in range(P)])
    ax.set_xticks([])
    ax.set_title(title)


P, M = 4, 8
fig, axes = plt.subplots(2, 1, figsize=(13, 4.5))
plot(gpipe_schedule(P, M),     axes[0], f"GPipe naive    (P={P}, M={M})")
plot(onefonefb_schedule(P, M), axes[1], f"1F1B           (P={P}, M={M})")
plt.tight_layout()
plt.show()

# Bubble fractions
bubble = (P - 1) / (M + P - 1)
print(f"\nGPipe bubble fraction: {bubble:.1%}")
print("1F1B has the same bubble but ~P× lower activation memory.")


## 4. Three-Dimensional Parallelism

For the largest models we combine all three axes. A 1024-GPU cluster might be laid out as

$$
\underbrace{8}_{\text{TP}} \,\times\, \underbrace{16}_{\text{PP}} \,\times\, \underbrace{8}_{\text{DP}} \,=\, 1024
$$

with TP within a node (NVLink), PP across nodes within a rack (NVSwitch / InfiniBand), DP across racks (InfiniBand). Each axis is assigned the interconnect that matches its frequency:

| Axis | Volume per step | Typical interconnect |
|------|-----------------|----------------------|
| TP   | High (per layer) | NVLink within a node |
| PP   | Medium (per stage boundary, micro-batch sized) | NVLink or InfiniBand |
| DP   | Low (one all-reduce per step, gradient-sized) | InfiniBand across nodes |

### A Megatron-LM Launch Configuration

```bash
torchrun --nproc_per_node=8 --nnodes=128 --rdzv_endpoint=master:29500 \
  pretrain_gpt.py \
  --tensor-model-parallel-size  8        \   # TP inside each node
  --pipeline-model-parallel-size 16      \   # PP across nodes
  --num-layers          80               \   # 80 layers, 16 pipeline stages → 5 per stage
  --hidden-size         12288            \
  --num-attention-heads 96               \
  --seq-length          2048             \
  --micro-batch-size    1                \
  --global-batch-size   1536             \
  --use-flash-attn                       \
  --bf16                                 \
  --recompute-granularity selective      \
  --tensorboard-dir ./tb
```

The product `tensor-model-parallel-size × pipeline-model-parallel-size × (world_size / both)` must equal `world_size`. Megatron derives the data-parallel size automatically: $\text{DP} = 1024 / (8 \cdot 16) = 8$.

### Selective Recompute

`--recompute-granularity selective` (Korthikanti et al., 2022) is a refinement of activation checkpointing: only the *cheap-to-recompute and large-in-memory* operations are recomputed (notably, the attention softmax and dropout). This recovers most of the memory savings of full checkpointing while spending only a fraction of the extra compute.


## 5. Pen-and-Paper: Sizing a Run

The exercise of **picking** the parallelism degrees is itself an HPC skill. The constraints are:

1. **Memory.** Each GPU must hold its share of model state + activations + KV-cache (for inference) + buffers.
2. **Throughput.** Communication must overlap compute. If you cannot hide an `all-reduce`, the run will be communication-bound.
3. **Batch size.** Global batch size $G = \text{DP} \cdot \text{micro} \cdot \text{grad\_accum}$. The model's *statistical* convergence behavior fixes $G$ within a band (too small → unstable; too large → wasteful).

A standard recipe for a model with $\Psi$ parameters across $N$ GPUs:

1. Choose **TP** such that the *largest single matmul fits comfortably*. TP = 8 on H100 covers up to ~30B-parameter dense layers.
2. Choose **PP** such that one stage's parameters + activations fit per GPU. Aim for 3–6 layers per stage.
3. Set **DP** = $N / (\text{TP} \cdot \text{PP})$. Adjust the micro-batch to keep the pipeline bubble below 15 %.

Below we estimate the bubble fraction across configurations.


In [ ]:
# Bubble-fraction calculator
def bubble_fraction(P, M):
    return (P - 1) / (M + P - 1)

print(f"{'P (pipeline)':>14} | {'M (micro)':>10} | {'Bubble':>8}")
print("-" * 40)
for P in (2, 4, 8, 16):
    for M in (4, 8, 16, 32, 64):
        print(f"{P:>14} | {M:>10} | {bubble_fraction(P, M):>8.1%}")
    print("-" * 40)


**Reading the table.** To keep the bubble below ~10 % with $P=16$, you need at least $M=64$ micro-batches per global batch. If your global batch is 1024 and TP × DP = 64, that gives $1024 / 64 = 16$ samples-per-pipeline — too few. The solution is either smaller global batch, more gradient accumulation, or interleaved 1F1B.


## 6. Reference Implementation Pointers

For your practical work this week:

- **Megatron-LM**: https://github.com/NVIDIA/Megatron-LM — the reference 3D-parallel implementation.
- **Megatron-DeepSpeed**: combines Megatron's TP/PP with DeepSpeed's ZeRO; used for training BLOOM and other open models.
- **PyTorch native**: `torch.distributed.tensor.parallel` (since PyTorch 2.0) implements column/row-parallel linears compatibly with FSDP.
- **DTensor**: PyTorch's distributed tensor abstraction, foundation of the new APIs.

The exercises below assume Megatron-LM as the target.


## 7. Exercises

**Exercise 5.1 — Communication budget.**  
For TP = 8 with a hidden dim of 12288, batch 8, sequence 2048, FP16: compute the bytes transferred per `all-reduce` and the time required on a 600 GB/s NVLink fabric. Compare to the compute time of a single transformer block.

**Exercise 5.2 — Pipeline scheduling.**  
Implement a function `schedule(P, M, mode)` that returns a 2D occupancy matrix for `mode ∈ {gpipe, 1f1b, interleaved}`. Visualize all three for $P=8, M=16$.

**Exercise 5.3 — Topology choice.**  
You have 256 H100 GPUs across 32 nodes of 8. Design TP/PP/DP for a 70B-parameter model (80 layers, hidden = 8192). Justify each choice by reference to interconnect bandwidth.

**Exercise 5.4 — Megatron config.**  
Read the launch script in `scripts/megatron/pretrain_demo.sh`. Identify every parallelism flag and explain its consequence in the per-GPU memory budget.

**Exercise 5.5 — Tensor parallel from scratch.**  
Using `torch.distributed`, run a 2-GPU job that creates `ColumnParallelLinear` and `RowParallelLinear` instances and verifies their output matches a single-GPU reference to FP16 precision.


## 8. Summary

Three orthogonal axes — data, tensor, and pipeline — let us split a trillion-parameter model across a thousand GPUs. Tensor parallelism shards the matmul within a layer and lives on NVLink. Pipeline parallelism shards layers across the cluster and tolerates slower interconnect. Data parallelism replicates and consumes the *cheapest* communication path. Choosing the right product of degrees is an engineering exercise governed by memory caps, the pipeline bubble, and the desired global batch size.

Week 6 turns to the *consumer* end of the stack: how do we *fine-tune* and *deploy* these models when our hardware budget is two orders of magnitude smaller than what trained them? LoRA, QLoRA, and modern quantization will let us close the gap.
